## Concept focus — Signals and graceful shutdown

Production scripts do not just need to start correctly; they need to stop correctly too. Signal handling is about understanding interruption as part of normal operation, especially in cron jobs, containers, and long-running workers.

```text
SIGINT / SIGTERM -> signal handler sets stop flag -> current unit finishes -> clean exit

unsafe pattern: signal handler does real work directly
```

### How to think about it
Treat a signal as a request, not a place to do the cleanup itself. The safe model is: handler records intent, main loop notices the flag, then normal program flow performs the shutdown steps deliberately.

### Visual references and further study
- [signal documentation](https://docs.python.org/3/library/signal.html)
- [subprocess documentation](https://docs.python.org/3/library/subprocess.html)
- [CLI guidelines](https://clig.dev/)
- [Python signal handling talk search](https://www.youtube.com/results?search_query=python+signal+handling)

---

# Module 25 — Automation, Scripting, and the OS

## Exercise 25.2 — Graceful shutdown, and why handlers must be trivial

Run:  python ex02_signals.py           then press Ctrl-C partway through
      python ex02_signals.py --unsafe  and press Ctrl-C repeatedly

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The anatomy of a robust script

In [ ]:
#!/usr/bin/env python3
"""One-line description. Longer explanation, and an example invocation."""

from __future__ import annotations

import logging
import signal
import sys
from pathlib import Path

logger = logging.getLogger(__name__)

EXIT_OK, EXIT_FAILURE, EXIT_USAGE, EXIT_INTERRUPTED = 0, 1, 2, 130


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    configure_logging(args.verbose)
    try:
        return run(args)
    except KeyboardInterrupt:
        logger.warning("interrupted")
        return EXIT_INTERRUPTED
    except AppError as exc:
        logger.error("%s", exc)
        return EXIT_FAILURE


if __name__ == "__main__":
    raise SystemExit(main())

Everything here has appeared before: `main(argv)` from Module 01, exit codes
from Module 07, exception handling from Module 16. What is new is treating them
as the *minimum* for anything another person or a scheduler will run.

**The five properties that make a script trustworthy:**

| Property | Why |
|---|---|
| **Idempotent** | Running it twice is harmless, so recovery is "run it again" |
| **Resumable** | It can continue after a partial failure |
| **Dry-runnable** | You can see what it would do before it does it |
| **Observable** | Its logs say what happened and how far it got |
| **Interruptible** | Ctrl-C stops it cleanly, without corrupting state |

**Idempotency is the most valuable of the five.** Every other property reduces
the chance of a bad outcome; idempotency makes recovery trivial.

---

## Concept 2. Signals and clean shutdown

In [ ]:
import signal

class GracefulExit:
    def __init__(self) -> None:
        self.should_stop = False
        signal.signal(signal.SIGTERM, self._handle)
        signal.signal(signal.SIGINT, self._handle)

    def _handle(self, signum: int, frame: object) -> None:
        logger.info("received %s, finishing the current item", signal.Signals(signum).name)
        self.should_stop = True

exit_flag = GracefulExit()
for item in items:
    process(item)
    if exit_flag.should_stop:
        logger.info("stopping cleanly after %s", item.id)
        break

**Set a flag; do not do the work in the handler.** A signal handler interrupts
the main thread at an arbitrary bytecode, so anything non-trivial there is a
race. Setting a boolean is safe; writing a file is not.

| Signal | Meaning | Catchable |
|---|---|---|
| `SIGINT` | Ctrl-C | yes |
| `SIGTERM` | Polite "please stop" — what Docker and systemd send | yes |
| `SIGKILL` | `kill -9` | **no** |
| `SIGHUP` | Terminal closed; conventionally "reload config" | yes |

Because `SIGKILL` cannot be caught, **your data must survive an abrupt death**
regardless — which is what atomic writes (Module 07) are for. Docker sends
`SIGTERM`, waits ten seconds, then `SIGKILL`; if your shutdown takes longer than
that grace period, you are being killed mid-work.

---

## Concept 6. Building a real CLI

`argparse` is in the standard library and sufficient. **Typer** (built on Click)
is what you would choose for anything with more than a few subcommands.

In [ ]:
import typer
app = typer.Typer(help="Sync files between two locations.")

@app.command()
def sync(
    source: Path = typer.Argument(..., exists=True, help="Source directory"),
    dest: Path = typer.Argument(..., help="Destination"),
    dry_run: bool = typer.Option(False, "--dry-run", help="Show what would happen"),
    workers: int = typer.Option(4, min=1, max=32),
) -> None:
    """Sync SOURCE to DEST."""

Typer derives parsing, validation, help text and shell completion **from the
type hints** (Module 17). `exists=True` on a `Path` means a missing directory is
a usage error with a clear message, not a traceback three functions later.

The CLI design rules that matter, from [clig.dev](https://clig.dev/):

- Data to stdout, messages to stderr (Module 07)
- `--json` for machine consumption
- Exit codes that mean something
- `--dry-run` on anything destructive
- Confirm before irreversible actions, with `--yes` to skip
- Show progress for anything over a few seconds
- Respect `NO_COLOR` and detect whether stdout is a TTY

---

## Concept 7. Scheduling

| Tool | Best for |
|---|---|
| `cron` | Simple recurring jobs on one machine |
| `systemd` timers | The same, with logging, dependencies and resource limits |
| APScheduler | In-process scheduling inside a long-running app |
| Celery beat / arq | Distributed scheduled tasks (Module 33) |
| Airflow / Dagster / Prefect | Data pipelines with dependencies and retries |

```cron
# m h dom mon dow  command
0 2 * * *  /opt/venv/bin/python /opt/jobs/nightly.py >> /var/log/nightly.log 2>&1
```

**The four things that break cron jobs, in order of frequency:**

1. **`PATH` is minimal.** Use absolute paths for the interpreter and every
   binary. `python` will not be found.
2. **The environment is not your shell's.** No `.bashrc`, no virtualenv, none of
   your exports. Set what you need explicitly in the crontab.
3. **The working directory is `$HOME`.** Relative paths resolve somewhere you
   did not expect.
4. **Output goes to mail nobody reads.** Redirect it to a file, or log properly.

Test with `env -i /usr/bin/env -` to reproduce cron's empty environment — that
one command finds most cron bugs before deployment.

`systemd` timers are better than cron for anything important: real logging via
journald, `OnFailure` hooks, dependency ordering, resource limits, and
`Persistent=true` so a missed run fires after a reboot.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The anatomy of a robust script
- Section 2: Signals and clean shutdown
- Section 3: Files, safely
- Section 4: `subprocess`, in practice
- Section 5: Configuration and environment
- Section 6: Building a real CLI
- Section 7: Scheduling
- Section 8: Watching the filesystem

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import json
import signal
import sys
import time
from pathlib import Path

STATE = Path("/tmp/ex02_state.json")


# TODO 1 -----------------------------------------------------------------------

---

## `GracefulExit`

Set a flag on SIGINT/SIGTERM. Nothing else.

In [ ]:
class GracefulExit:
    """Set a flag on SIGINT/SIGTERM. Nothing else.

    Requirements:
      - handle both signals
      - record WHICH signal arrived
      - a SECOND signal should exit immediately (a user pressing Ctrl-C twice
        means "I meant it") -- decide what "immediately" should do about
        in-flight work, and justify it
      - restore the previous handlers on exit, so this is usable as a context
        manager inside a larger program
    """

---

## `process_items`

Process items, checking the flag BETWEEN items, never during one.

In [ ]:
def process_items(items: list[int], exit_flag) -> int:  # type: ignore[no-untyped-def]
    """Process items, checking the flag BETWEEN items, never during one.

    Each item must be committed atomically before moving on, so an interruption
    leaves a consistent state. Log progress every N items so an operator can
    see where it stopped.
    """
    raise NotImplementedError

---

## `save_checkpoint`

Atomically record how far we got (Module 07).

In [ ]:
def save_checkpoint(index: int) -> None:
    """Atomically record how far we got (Module 07)."""
    raise NotImplementedError

---

## `load_checkpoint`

Return the resume point, or 0.

In [ ]:
def load_checkpoint() -> int:
    """Return the resume point, or 0."""
    raise NotImplementedError

---

## `unsafe_handler`

DO NOT DO THIS. It is here so you can watch it fail.

In [ ]:
def unsafe_handler(signum: int, frame: object) -> None:
    """DO NOT DO THIS. It is here so you can watch it fail.

    Does real work in the handler: writes a file, sleeps, prints.

    Run with --unsafe and press Ctrl-C several times in quick succession.
    Record what happens. You are looking for at least one of:
      - a handler re-entered while still running
      - a partially written file
      - a traceback from inside the handler
      - output interleaved mid-line

    Then answer: WHY is this unsafe? The handler runs in the main thread,
    between two bytecodes, at a point you did not choose. Name three specific
    things that could be half-finished at that moment.
    """
    print("  handler: starting work...")
    STATE.write_text(json.dumps({"handler": "ran", "at": time.time()}))
    time.sleep(0.5)
    print("  handler: done")

---

## `with_deadline`

Shutdown with a deadline.

In [ ]:
def with_deadline(exit_flag, seconds: float) -> None:  # type: ignore[no-untyped-def]
    """Shutdown with a deadline.

    After the flag is set, finish the current item but give up entirely if
    cleanup exceeds `seconds` -- because a container's grace period is finite
    (Docker: 10s, Kubernetes: 30s by default) and exceeding it means SIGKILL
    mid-work anyway.

    Then answer: how would you find out your real grace period, and what should
    a service do if its clean shutdown genuinely needs longer than it?
    """
    raise NotImplementedError

---

## `main`

_main_

In [ ]:
def main(argv: list[str]) -> int:
    if "--unsafe" in argv:
        signal.signal(signal.SIGINT, unsafe_handler)
        print("UNSAFE mode. Press Ctrl-C repeatedly and watch.")
        for i in range(60):
            print(f"  working {i}", flush=True)
            time.sleep(0.2)
        return 0

    print("Press Ctrl-C to stop cleanly. Then run again to see it resume.")
    raise NotImplementedError("implement the TODOs")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    raise SystemExit(main(sys.argv[1:]))

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.